In [89]:
import kagglehub
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("carrie1/ecommerce-data")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ecommerce-data' dataset.
Path to dataset files: /kaggle/input/ecommerce-data


In [90]:
df = pd.read_csv(path + "/data.csv", encoding = "ISO-8859-1")

In [91]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [92]:
df.sample(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
527537,580724,23389,SPACEBOY MINI BACKPACK,1,12/5/2011 17:09,4.15,NaN,United Kingdom
441690,574624,23531,WALL ART BIG LOVE,6,11/6/2011 11:20,6.95,17769.0,United Kingdom
485562,577609,23108,SET OF 10 LED DOLLY LIGHTS,2,11/21/2011 9:51,6.25,12349.0,Italy
342112,566818,23359,SET OF 12 T-LIGHTS VINTAGE DOILY,24,9/15/2011 11:20,1.95,13104.0,United Kingdom
465409,576228,21918,SET 12 KIDS COLOUR CHALK STICKS,6,11/14/2011 13:12,0.42,15033.0,United Kingdom


# E-Commerce Data

Este dataset que **simula o dia a dia real de uma empresa**. É um pouco "sujo", exige cálculos de dinheiro e datas, e é o tipo de teste técnico que empresas podem aplicar.

## O Dataset: E-Commerce Data (Transações Reais)

Este dataset contém transações de uma loja de varejo online baseada no Reino Unido.

**Onde baixar:** [Kaggle - E-Commerce Data](https://www.kaggle.com/carrie1/ecommerce-data)

**Estrutura Importante:**
*   `InvoiceNo`: Número da fatura (Se começar com 'C', é um cancelamento/devolução).
*   `StockCode`: Código do produto.
*   `Description`: Nome do produto.
*   `Quantity`: Quantidade comprada (pode ser negativo se for devolução).
*   `InvoiceDate`: Data e hora da compra.
*   `UnitPrice`: Preço unitário.
*   `CustomerID`: ID do cliente.
*   `Country`: País.

---

## Briefing

A diretoria precisa de um relatório de performance de vendas. O problema é que os dados brutos contêm devoluções (números negativos) que podem estragar as somas. Você precisa limpar isso antes de calcular.

---

### Dica para a Q1 (Encoding)
Esse arquivo CSV específico às vezes dá erro de leitura por causa de caracteres especiais. Se o `pd.read_csv` der erro de `UnicodeDecodeError`, use:
```python
df = pd.read_csv("caminho/data.csv", encoding="ISO-8859-1")
```


1.  **Limpeza de Dados (Sanity Check):**
    *   A empresa só quer saber de **vendas efetivas**. Remova do DataFrame todas as linhas onde `Quantity` for menor ou igual a 0.
    *   Além disso, remova linhas onde `CustomerID` é nulo (NaN), pois precisamos identificar os clientes.
    *   *Dica:* Faça isso logo no começo e use esse dataframe limpo para as próximas questões. Quantas linhas sobraram? (Imprima o `shape` ou `len` no final).



In [93]:
print(f"DF Original: {df.shape[0]}")

df.dropna(subset = ["CustomerID"], inplace = True)

print(f"Droped 'Customer' (no Customer NaNs): {df.shape[0]}")

print(f"Quantity of <=0 entries in 'Quantity' column (sum): {df[df["Quantity"] <= 0].value_counts().sum()}")

mascara = df["Quantity"] <= 0

df = df[~mascara]

print(f"Droped Quantity column (no zero and negatives): {df.shape[0]}")

print(f"check of 'Quantity' column <=: {df[df["Quantity"] <= 0].value_counts().sum()}")

print(f"df.shape: {df.shape}")

DF Original: 541909
Droped 'Customer' (no Customer NaNs): 406829
Quantity of <=0 entries in 'Quantity' column (sum): 8905
Droped Quantity column (no zero and negatives): 397924
check of 'Quantity' column <=: 0
df.shape: (397924, 8)


2.  **Engenharia de Atributos (Faturamento):**
    *   O dataset tem `Quantity` e `UnitPrice`, mas não tem o valor total da compra.
    *   Crie uma nova coluna chamada `TotalAmount` que seja a multiplicação de Quantidade por Preço Unitário.
    *   **Pergunta:** Qual foi o valor (`TotalAmount`) da venda única mais cara registrada no sistema? (Armazene o valor float em `max_order_value`).



In [94]:
df["TotalAmount"]  = df["Quantity"] * df["UnitPrice"]

max_order_value = df["TotalAmount"].max()

max_order_value

168469.6

3.  **Análise Temporal (Melhor Mês):**
    *   A coluna `InvoiceDate` provavelmente virá como texto. Converta-a para datetime (`pd.to_datetime`).
    *   **Pergunta:** Qual foi o mês com o maior faturamento total (`TotalAmount`)?
    *   *Dica:* Extraia o mês (ou Ano-Mês) e some o `TotalAmount`.
    *   Armazene o resultado (ex: número do mês ou string "2011-11") na variável `best_month`.



In [95]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
# df.info()
# df.reset_index()
# df = df.set_index('InvoiceDate')
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["Day"] = df["InvoiceDate"].dt.day



df.head()
best_month = df.groupby(["Year", "Month"])["TotalAmount"].sum()
best_Y, best_M = best_month.idxmax()
best_month = str(best_Y) + "-" + str(best_M)

print(best_month)
type(best_month)

2011-11


str

4.  **Pareto e Clientes VIP:**
    *   Queremos saber quem gasta mais. Agrupe os dados por `CustomerID` e some o `TotalAmount`.
    *   **Pergunta:** Qual é o `CustomerID` do cliente que mais gastou dinheiro na loja em todo o período?
    *   Armazene o ID (geralmente é um float ou int) na variável `top_customer_id`.




In [140]:
# df.reset_index()
# df = df.set_index("CustomerID")
top_customer_id = df.groupby(["CustomerID"])["TotalAmount"].sum().idxmax()
# top_customer_id = top_customer_id.sort_values(ascending=False)
# top_customer_id = df.loc[top_customer_id]
top_customer_id = int(top_customer_id)

print(top_customer_id)

14646


In [122]:
# df.loc[top_customer_id]

5.  **Produtos Exclusivos:**
    *   Olhando apenas para este cliente VIP (`top_customer_id`) que você descobriu na Q4: Quantos itens **diferentes** (unique `StockCode`) ele comprou?
    *   *Atenção:* Não queremos a quantidade de peças, mas a variedade de produtos (ex: comprou 100 itens, mas eram todos do mesmo código, então a resposta é 1).
    *   Armazene esse número inteiro em `vip_unique_items`.

In [141]:
x = df[df["CustomerID"] == top_customer_id]
x.shape

(2080, 12)

In [138]:
# vip_unique_items = df.groupby(top_customer_id)["Stock_Code"].value_counts().unique()
# vip_unique_items = df[df["CustomerID"] == top_customer_id]

vip_unique_items = df[df["CustomerID"] == top_customer_id]["StockCode"].unique()
vip_unique_items = vip_unique_items.shape[0]
vip_unique_items

701